# Capítulo 14 — De un fenómeno a un modelo

**Cuaderno interactivo de *La servilleta y el ordenador*.**

Cada sección reproduce una figura del capítulo. La gracia no es ejecutarlas: es **cambiar los parámetros y comprobar si ocurre lo que esperabas**.

> Antes de ejecutar cada celda, escribe en una línea qué esperas ver. Después mira si ocurrió. Y después, por qué.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() / ''))
sys.path.insert(0, '../../../herramientas')
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / 'herramientas'))

import numpy as np
import matplotlib.pyplot as plt
from estilo_libro import C, use_style, rng, save

use_style()
%matplotlib inline

---

## La taza de café, cuarta visita: cuando el modelo mínimo no basta.

Datos sintéticos realistas de enfriamiento con evaporación. Ajuste con el
modelo de Newton, sus residuos, y el modelo mejorado.

La figura responde: ¿cómo se sabe que hace falta más modelo, y cuánto más?

Ejecutar:  python fig_cafe_progresivo.py

*(script original: `codigo/fig_cafe_progresivo.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

from estilo_libro import C, rng, save, use_style  # noqa: E402

use_style()
r = rng(140)

T_AMB = 21.0
t = np.arange(0, 121, 3.0)

# "Verdad" sintética: convección lineal MÁS evaporación, que depende de forma
# no lineal del salto térmico. Coeficientes en K/min, ajustados para reproducir
# un enfriamiento realista (92 -> 70 °C en unos 15 minutos).
TAU_CONV = 32.0          # min, término de Newton puro
K_EVAP = 3.0e-4          # K^{-0.9} min^{-1}, evaporación


def verdad(t):
    """Integra la EDO 'real' con paso fino y devuelve los valores en t."""
    dt = 0.02
    Ti, tt, idx = 92.0, 0.0, 0
    salida = []
    while idx < len(t):
        if tt >= t[idx] - 1e-9:
            salida.append(Ti)
            idx += 1
            continue
        d = Ti - T_AMB
        Ti += (-d / TAU_CONV - K_EVAP * d**1.9) * dt
        tt += dt
    return np.array(salida)


T_datos = verdad(t) + r.normal(0, 0.35, t.size)


def newton(t, T0, tau):
    return T_AMB + (T0 - T_AMB) * np.exp(-t / tau)


def dos_exp(t, T0, tau1, f, tau2):
    a = (T0 - T_AMB)
    return T_AMB + a * (f * np.exp(-t / tau1) + (1 - f) * np.exp(-t / tau2))


p1, _ = curve_fit(newton, t, T_datos, p0=[92, 25])
p2, _ = curve_fit(dos_exp, t, T_datos, p0=[92, 10, 0.4, 40],
                  maxfev=40000)
res1 = T_datos - newton(t, *p1)
res2 = T_datos - dos_exp(t, *p2)

fig, axes = plt.subplots(2, 2, figsize=(10.4, 6.0), sharex=True,
                         gridspec_kw={"height_ratios": [1.7, 1], "hspace": 0.12})

for col, (nombre, modelo, p, res) in enumerate([
        ("Modelo mínimo: Newton", newton, p1, res1),
        ("Dos escalas de tiempo", dos_exp, p2, res2)]):
    ax = axes[0, col]
    ax.plot(t, T_datos, "o", color=C.red, ms=3.5, label="datos")
    tt = np.linspace(0, 120, 400)
    ax.plot(tt, modelo(tt, *p), color=C.blue, lw=1.8, label="ajuste")
    ax.axhline(T_AMB, color=C.grey, ls="--", lw=1.0)
    ax.set_ylabel("temperatura (°C)")
    rms = np.std(res)
    ax.set_title(f"{nombre}\nrms de los residuos = {rms:.2f} °C", fontsize=10)
    ax.legend(fontsize=8)

    ax = axes[1, col]
    ax.axhline(0, color=C.ink, lw=1.1)
    ax.axhspan(-0.35, 0.35, color=C.grey, alpha=0.2)
    ax.plot(t, res, "o-", color=C.ochre if col == 0 else C.green, ms=3.5, lw=1)
    ax.set_xlabel("tiempo (min)"), ax.set_ylabel("residuo (°C)")
    ax.set_ylim(-2.2, 2.2)
    print(f"{nombre:26s} rms={rms:.3f} °C   parámetros={np.round(p,3)}")

axes[1, 0].text(35, 1.35, "estructura clara:\nel modelo está incompleto",
                fontsize=8.6, color=C.red)
axes[1, 1].text(35, 1.35, "ruido: el modelo agota\nla información de los datos",
                fontsize=8.6, color=C.green)
axes[1, 0].text(3, -2.0, "banda gris: ruido de medida declarado (0,35 °C)",
                fontsize=7.6, color=C.grey)

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## El ciclo completo del modelador, en un diagrama.

Diagrama conceptual de las quince etapas, con las preguntas asociadas a cada
una y los bucles de realimentación.

La figura responde: ¿en qué orden se hacen las cosas, y dónde se vuelve atrás?

Ejecutar:  python fig_ciclo.py

*(script original: `codigo/fig_ciclo.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

from estilo_libro import C, caja, flecha, lienzo, save, use_style  # noqa: E402

use_style()
fig, ax = lienzo(ancho=10.5, alto=6.2, xlim=(0, 14), ylim=(1.6, 10))
ax.set_aspect("auto")

ETAPAS = [
    (2.2, 9.2, "FENÓMENO", "algo que observas", C.ink),
    (2.2, 8.0, "Pregunta", "¿qué quiero saber,\ncon qué precisión?", C.blue),
    (2.2, 6.8, "Orden de magnitud", "¿qué número espero?", C.blue),
    (2.2, 5.6, "Variables", "¿de qué depende?\n¿de qué NO?", C.blue),
    (2.2, 4.4, "Supuestos", "escritos, numerados,\ncon su condición", C.blue),
    (2.2, 3.2, "Modelo mínimo", "lo más simple que\npodría funcionar", C.blue),
    (7.0, 3.2, "Ecuaciones", "y su análisis\nde escalas", C.green),
    (7.0, 4.4, "Solución aproximada", "límites, casos\nextremos", C.green),
    (7.0, 5.6, "Simulación", "predicción escrita\nANTES", C.green),
    (7.0, 6.8, "Validación", "¿contra qué dato?", C.ochre),
    (7.0, 8.0, "Incertidumbre", "¿cuánto me fío?", C.ochre),
    (11.8, 8.0, "Interpretación", "¿qué significa?", C.red),
    (11.8, 6.8, "Límites", "¿dónde deja\nde valer?", C.red),
    (11.8, 5.6, "NUEVA PREGUNTA", "y vuelta a empezar", C.ink),
]

for x, y, titulo, sub, color in ETAPAS:
    destacado = titulo.isupper()
    caja(ax, x, y, 3.5, 0.95,
         f"{titulo}\n{sub}",
         color=color, fontsize=8.2,
         relleno="#f2f5f9" if destacado else "white", lw=2.0 if destacado else 1.3)

# Flujo principal
for i in range(5):
    flecha(ax, (2.2, ETAPAS[i][1] - 0.5), (2.2, ETAPAS[i + 1][1] + 0.5),
           color=C.grey, lw=1.3)
flecha(ax, (3.95, 3.2), (5.25, 3.2), color=C.grey, lw=1.3)
for i in range(6, 10):
    flecha(ax, (7.0, ETAPAS[i][1] + 0.5), (7.0, ETAPAS[i + 1][1] - 0.5),
           color=C.grey, lw=1.3)
flecha(ax, (8.75, 8.0), (10.05, 8.0), color=C.grey, lw=1.3)
flecha(ax, (11.8, 7.5), (11.8, 7.3), color=C.grey, lw=1.3)
flecha(ax, (11.8, 6.3), (11.8, 6.1), color=C.grey, lw=1.3)

# Bucles de realimentación
flecha(ax, (10.05, 5.6), (3.95, 4.4), color=C.red, lw=1.6, rad=-0.25)
ax.text(7.0, 2.05, "si el modelo falla: vuelve a los supuestos, no a las "
        "ecuaciones", fontsize=8.4, color=C.red, ha="center")
flecha(ax, (5.25, 6.8), (3.95, 6.8), color=C.ochre, lw=1.4, rad=0.0,
       texto="¿coincide con\nla estimación?", fontsize=7.6, desplaza=(0, 0.62))

ax.text(0.15, 1.75, "Las tres primeras etapas y las tres últimas distinguen a "
        "un modelador de alguien que sabe resolver ecuaciones.",
        fontsize=9.0, color=C.ink, style="italic")

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 
